In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
select * from fk_ce1366.bronze.orders limit 1

customer_id,delivery,items,order_id,order_status,order_ts,payment,store_id,updated_at,_rescued_data,_source_file,ingested_at
C0383,"List(21, 0.0, 15)","List(List(0.0, P006, 3, 35.0), List(63.0, P044, 4, 85.0), List(26.0, P002, 1, 159.0), List(0.0, P034, 1, 20.0))",FK-20260304-0126,DELIVERED,2026-03-04 09:39:57,"List(Card, TXN3599496630)",S10,2026-03-04 10:00:57,null,/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-04.json,2026-10-12T11:50:04.854Z


In [0]:
%sql
select * from fk_ce1366.silver.orders limit 1

customer_id,items,order_id,order_status,order_ts,store_id,updated_at,order_date,payment_method,promised_mins,actual_mins,delivery_fee
C0383,"List(List(0.0, P006, 3, 35.0), List(63.0, P044, 4, 85.0), List(26.0, P002, 1, 159.0), List(0.0, P034, 1, 20.0))",FK-20260304-0126,DELIVERED,2026-03-04T09:39:57.000Z,S10,2026-03-04T10:00:57.000Z,2026-03-04,Card,15,21,0.00


In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("store_id", col("store_id").cast("string"))
    .withColumn("order_ts", try_to_timestamp(col("order_ts"), lit("yyyy-MM-dd HH:mm:ss")))
    .withColumn("order_date", to_date(col("order_ts")))
    .withColumn("order_status", trim(upper(col("order_status"))).cast("string"))
    .withColumn("payment_method", trim(upper(col("payment.method"))).cast("string"))
    .withColumn("channel", lit("Unknown"))
    .withColumn("promised_mins", col("delivery.promised_mins").cast("int"))
    .withColumn("actual_mins", col("delivery.actual_mins").cast("int"))
    .withColumn("delivery_fee", col("delivery.delivery_fee").cast("decimal(10,2)"))
    .withColumn("updated_at", try_to_timestamp(col("updated_at")))
)

df = df.drop(
    "ingested_at",
    "_source_file",
    "payment",
    "delivery",
    "items",
    "_rescued_data"
)

df.write.option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{catalog}.silver.orders")

In [0]:
from pyspark.sql.functions import *

df = spark.read.table(f"{catalog}.bronze.orders")

df = df.withColumn("item", explode(col("items"))).drop("items")

df = (
    df.withColumn("order_id", col("order_id").cast("string"))
    .withColumn("product_id", col("item.product_id").cast("string"))
    .withColumn(
        "quantity",
        when(col("item.qty") > 0, col("item.qty")).otherwise(lit(None))
    )
    .withColumn(
        "unit_price",
        when(col("item.unit_price") > 0, col("item.unit_price")).otherwise(lit(None))
    )
    .withColumn(
        "discount_amount",
        when(col("item.discount") >= 0, col("item.discount"))
        .otherwise(lit(None))
    )
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn(
        "net_amount",
        col("gross_amount") - coalesce(col("discount_amount"), lit(0))
    )
)

df = df.drop(
    "ingested_at",
    "_rescued_data",
    "_source_file",
    "payment",
    "delivery",
    "item",
    "updated_at",
    "order_ts",
    "order_status",
    "payment_method",
    "customer_id",
    "store_id"
)

df.write.option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.order_items")

In [0]:
from pyspark.sql import Window

def duplicate_order_ids():
    df = spark.read.table(f"{catalog}.bronze.orders")
    window = Window.partitionBy("order_id").orderBy(desc("updated_at"))
    df = df.withColumn("rn", row_number().over(window))
    df = df.filter(col("rn") == 1).drop("rn")
    return df

d = duplicate_order_ids()
d.count()

1084

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, broadcast

orders_df = duplicate_order_ids()

orders_df = (orders_df
    .withColumn("raw_record", F.to_json(F.struct(*orders_df.columns)))
    .withColumn("order_id", col("order_id").cast("string")))

valid_customers = (spark.read.table(f"{catalog}.silver.customers")
    .select(col("customer_id").cast("string").alias("valid_customer_id")).distinct())
valid_stores = (spark.read.table(f"{catalog}.silver.stores")
    .select(col("store_id").cast("string").alias("valid_store_id")).distinct())
valid_products = (spark.read.table(f"{catalog}.silver.products")      # active + inactive
    .select(col("product_id").cast("string").alias("valid_product_id")).distinct())

lines_df = (
    orders_df
    .filter(col("items").isNotNull() & (F.size("items") > 0))
    .select(col("order_id"), F.explode("items").alias("item"))
    .select(
        col("order_id"),
        col("item.product_id").cast("string").alias("line_product_id"),
        col("item.qty").alias("line_qty"),                 
        col("item.unit_price").alias("line_price"),
    )
    .join(broadcast(valid_products),
          col("line_product_id") == col("valid_product_id"), "left")
)

line_flags = (
    lines_df
    .withColumn("bad_product_i",
        (col("line_product_id").isNull() | col("valid_product_id").isNull()).cast("int"))
    # S7 (assumed): quantity must be a whole number > 0
    .withColumn("qty_int", F.expr("try_cast(line_qty as int)"))
    .withColumn("bad_qty_i",
        (col("qty_int").isNull() | (col("qty_int") <= 0)).cast("int"))
    # S8 (assumed): unit_price must be a number >= 0
    .withColumn("price_dec", F.expr("try_cast(line_price as decimal(18,2))"))
    .withColumn("bad_price_i",
        (col("price_dec").isNull() | (col("price_dec") < 0)).cast("int"))
    .groupBy("order_id")
    .agg(
        (F.max("bad_product_i") == 1).alias("bad_product"),
        (F.max("bad_qty_i") == 1).alias("bad_qty"),
        (F.max("bad_price_i") == 1).alias("bad_price"),
    )
)

ts_valid = F.coalesce(
    F.expr("try_to_timestamp(order_ts, 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(order_ts, \"yyyy-MM-dd'T'HH:mm:ss\")")
).isNotNull()

flagged = (
    orders_df
    .join(broadcast(valid_customers),
          col("customer_id").cast("string") == col("valid_customer_id"), "left")
    .join(broadcast(valid_stores),
          col("store_id").cast("string") == col("valid_store_id"), "left")
    .join(line_flags, "order_id", "left")
)

reason = (
    F.when(col("customer_id").isNull(), "MISSING_CUSTOMER")                             
     .when(col("valid_customer_id").isNull(), "UNKNOWN_CUSTOMER")                      
     .when(col("store_id").isNull() | col("valid_store_id").isNull(), "UNKNOWN_STORE")   
     .when(col("order_ts").isNull() | ~ts_valid, "INVALID_TIMESTAMP")                    
     .when(col("items").isNull() | (F.size("items") == 0), "EMPTY_ORDER")                
     .when(col("bad_product"), "UNKNOWN_PRODUCT")                                       
     .when(col("bad_qty"), "INVALID_QUANTITY")                                         
     .when(col("bad_price"), "INVALID_PRICE")                                           
)

quarantine_orders_df = (
    flagged
    .withColumn("reject_reason", reason)
    .filter(col("reject_reason").isNotNull())
    .select(
        col("order_id").cast("string").alias("order_id"),
        col("reject_reason"),
        F.expr("try_to_timestamp(updated_at)").alias("updated_at"),
        col("raw_record"),
        F.current_timestamp().alias("quarantined_at"),
    )
)

(quarantine_orders_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.quarantine_orders"))

quarantine_orders_df.display()

order_id,reject_reason,updated_at,raw_record,quarantined_at
FK-20260301-0003,UNKNOWN_CUSTOMER,2026-03-01T18:45:39.000Z,"{""customer_id"":""C0206 "",""delivery"":{""actual_mins"":9,""delivery_fee"":0.0,""promised_mins"":10},""items"":[{""discount"":0.0,""product_id"":""P049"",""qty"":1,""unit_price"":128.0},{""product_id"":""P021"",""qty"":1,""unit_price"":279.0}],""order_id"":""FK-20260301-0003"",""order_status"":""DELIVERED"",""order_ts"":""2026-03-01 18:36:39"",""payment"":{""method"":""UPI "",""txn_ref"":""TXN7787137990""},""store_id"":""S03"",""updated_at"":""2026-03-01 18:45:39"",""_source_file"":""/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-01.json"",""ingested_at"":""2026-10-12T11:50:04.854Z""}",2026-10-13T12:04:16.437Z
FK-20260301-0011,UNKNOWN_PRODUCT,2026-03-01T16:35:22.000Z,"{""customer_id"":""C0372"",""delivery"":{""actual_mins"":12,""delivery_fee"":0.0,""promised_mins"":15},""items"":[{""discount"":14.0,""product_id"":""p057"",""qty"":1,""unit_price"":99.0},{""discount"":0.0,""product_id"":""P002"",""qty"":1,""unit_price"":159.0}],""order_id"":""FK-20260301-0011"",""order_status"":""DELIVERED"",""order_ts"":""2026-03-01 16:23:22"",""payment"":{""method"":""card"",""txn_ref"":""TXN8788195942""},""store_id"":""S12"",""updated_at"":""2026-03-01 16:35:22"",""_source_file"":""/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-01.json"",""ingested_at"":""2026-10-12T11:50:04.854Z""}",2026-10-13T12:04:16.437Z
FK-20260301-0019,UNKNOWN_PRODUCT,2026-03-01T12:30:25.000Z,"{""customer_id"":""C0156"",""delivery"":{""actual_mins"":12,""delivery_fee"":0.0,""promised_mins"":15},""items"":[{""product_id"":""P053"",""qty"":1,""unit_price"":175.0},{""discount"":0.0,""product_id"":""p010"",""qty"":1,""unit_price"":24.0}],""order_id"":""FK-20260301-0019"",""order_status"":""DELIVERED"",""order_ts"":""2026-03-01 12:18:25"",""payment"":{""method"":""UPI"",""txn_ref"":""TXN1209525095""},""store_id"":""S12"",""updated_at"":""2026-03-01 12:30:25"",""_source_file"":""/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-01.json"",""ingested_at"":""2026-10-12T11:50:04.854Z""}",2026-10-13T12:04:16.437Z
FK-20260301-0021,UNKNOWN_PRODUCT,2026-03-02T15:38:16.000Z,"{""customer_id"":""C0291"",""delivery"":{""actual_mins"":7,""delivery_fee"":0.0,""promised_mins"":10},""items"":[{""product_id"":""p055"",""qty"":2,""unit_price"":355.0},{""discount"":0.0,""product_id"":""P057"",""qty"":1,""unit_price"":99.0}],""order_id"":""FK-20260301-0021"",""order_status"":""Delivered "",""order_ts"":""2026-03-01 14:40:16"",""payment"":{""method"":""CARD"",""txn_ref"":""TXN3490499829""},""store_id"":""S05"",""updated_at"":""2026-03-02 15:38:16"",""_source_file"":""/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-02.json"",""ingested_at"":""2026-10-12T11:50:04.854Z""}",2026-10-13T12:04:16.437Z
FK-20260301-0022,UNKNOWN_PRODUCT,2026-03-01T17:31:23.000Z,"{""customer_id"":""C0187"",""delivery"":{""actual_mins"":13,""delivery_fee"":0.0,""promised_mins"":15},""items"":[{""product_id"":""p021"",""qty"":1,""unit_price"":279.0}],""order_id"":""FK-20260301-0022"",""order_status"":""DELIVERED"",""order_ts"":""2026-03-01 17:18:23"",""payment"":{""method"":"" Upi"",""txn_ref"":""TXN3127678210""},""store_id"":""S02"",""updated_at"":""2026-03-01 17:31:23"",""_source_file"":""/Volumes/fk_ce1366/landing/raw/orders/orders_2026-03-01.json"",""ingested_at"":""2026-10-12T11:50:04.854Z""}",2026-10-13T12:04:16.437Z
FK-20260301-0079,INVALID_QUANTITY,2026-03-01T14:58:46.000Z,"{""customer_id"":""C0343"",""delivery"":{""actual_mins"":6,""delivery_fee"":25.0,""promised_mins"":10},""items"":[{""discount"":0.0,""product_id"":""P043"",""qty"":0,""unit_price"":45.0},{""discount"":0.0,""product_id"":""P041"",""qty"":1,""unit_price"":132.0}],""order_id"":""FK-20260301-0079"",""order_status"":""delivered"",""order_ts"":""2026-03-01 14:52:46"",""payment"":{""method"":""UPI"",""txn_ref"":""TXN6451295527""},""store_id"":""S01"",""updated_at"":""2026-03-01 14:58:46"",""_source_file"":""/Volumes